# Module 3.3 - Knowledge Graphs

**Reference:** [`03-knowledge-graphs.md`](./03-knowledge-graphs.md)

## What you'll do in this notebook

- Build a small directed knowledge graph with NetworkX.
- Extract entities and relationships from free text using the LLM.
- Combine vector search (for 'what does this passage say?') with graph traversal (for 'who is connected to whom?').

> NetworkX is in `requirements.txt`. A production system would use a proper graph database like Neo4j - the concepts here transfer directly.

## Setup

In [ ]:
import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI
import networkx as nx

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Build a graph by hand

Use `nx.DiGraph` (directed graph). Add nodes for entities and edges carrying a `relation` attribute for each relationship.

In [ ]:
G = nx.DiGraph()

facts = [
    ("Python", "created_by", "Guido van Rossum"),
    ("Python", "released_in", "1991"),
    ("Django", "built_with", "Python"),
    ("Django", "used_for", "web development"),
    ("Flask", "built_with", "Python"),
    ("Flask", "used_for", "web development"),
    ("Guido van Rossum", "worked_at", "Google"),
    ("Guido van Rossum", "worked_at", "Dropbox"),
    ("JavaScript", "created_by", "Brendan Eich"),
    ("JavaScript", "released_in", "1995"),
    ("React", "built_with", "JavaScript"),
]

# TODO: add every (subject, relation, object) triple from `facts` to G using
# G.add_edge(subject, object, relation=relation).

print(f"Graph has {G.number_of_nodes()} entities and {G.number_of_edges()} relationships")

## Exercise 2 - Graph queries

Traversal is what makes graphs useful. Write helpers that answer direct and multi-hop questions.

In [ ]:
def neighbours_by_relation(entity: str, relation: str) -> list[str]:
    """Every node x such that (entity) --relation--> (x)."""
    # TODO: iterate over G.successors(entity) and keep those whose edge
    # relation == relation.
    raise NotImplementedError

def describe(entity: str) -> list[tuple[str, str, str]]:
    """Every outgoing triple from entity, one hop only."""
    # TODO: for each successor s of entity, return (entity, relation, s).
    raise NotImplementedError

print("Python was created by:", neighbours_by_relation("Python", "created_by"))
print("Guido worked at:   ", neighbours_by_relation("Guido van Rossum", "worked_at"))

print("\nAll facts about Python:")
for s, r, o in describe("Python"):
    print(f"  ({s}) --{r}--> ({o})")

**Multi-hop challenge:** write a query that answers *"Which frameworks are built with the language created by Guido van Rossum?"* — you'll need two hops through the graph.

In [ ]:
# TODO: starting from "Guido van Rossum", find the language he created,
# then find every framework that is built_with that language. Print them.


## Exercise 3 - Automatic entity + relation extraction

Building a graph by hand is fine for a handful of facts. For real documents, let the LLM extract triples. The key is asking for strictly structured output - JSON here - so you can parse reliably.

In [ ]:
EXTRACT_SYSTEM = (
    "You extract factual (subject, relation, object) triples from text. "
    "Reply with STRICT JSON of the form {\"triples\": [[\"Subject\", \"relation\", \"Object\"], ...]}. "
    "Use lowercase_snake_case for relations. No commentary."
)

def extract_triples(text: str) -> list[tuple[str, str, str]]:
    # TODO:
    # 1. Call the chat API with system=EXTRACT_SYSTEM and user=text, temperature=0,
    #    and response_format={"type": "json_object"}.
    # 2. json.loads the reply and return [(s, r, o) for s, r, o in payload["triples"]].
    raise NotImplementedError

PASSAGE = (
    "Rust was designed at Mozilla Research by Graydon Hoare. The first stable "
    "release, Rust 1.0, came out in 2015. Cargo is the official package manager "
    "for Rust."
)

for s, r, o in extract_triples(PASSAGE):
    print(f"  ({s}) --{r}--> ({o})")

In [ ]:
# TODO: add the extracted triples into the graph G with G.add_edge, then
# run describe("Rust") to confirm the new facts are there.


## Exercise 4 - Route questions to graph or vector search

In a real hybrid system you decide per query whether to use graph reasoning, vector search, or both. A simple heuristic: questions that start with "who", "when", "where" often map to graph queries; "how", "why", "explain" usually map to vector search.

In [ ]:
GRAPH_KEYWORDS = ("who", "when", "where", "which", "related to", "connected to")

def needs_graph(question: str) -> bool:
    # TODO: return True if any entry from GRAPH_KEYWORDS appears in question.lower().
    raise NotImplementedError

for q in [
    "Who created Python?",
    "How do I handle errors in Python?",
    "When was JavaScript released?",
    "Explain Python list comprehensions.",
]:
    print(f"[{'graph' if needs_graph(q) else 'vector'}] {q}")

## What's next

Every fancy retrieval technique costs more API calls. `04-semantic-caching.ipynb` shows how to avoid redoing work by caching responses for queries the system has already answered.